# Evaluate

In this notebook we evaluate the accuracy of the predicted alignments.

In [1]:
%matplotlib inline

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib import cm
import seaborn as sns
from pathlib import Path
import glob
import os.path
import pandas as pd
import pickle
import re
from plotnine import *
from pandas.api.types import CategoricalDtype

In [3]:
DATASET = 'test' # 'test'
VERSION = 'full'

In [4]:
ANNOTATIONS_ROOT = Path('/home/ijain/ttmp/Chopin_Mazurkas/annotations_beat')
query_list = Path(f'cfg_files/queries.{DATASET}.{VERSION}')

### Evaluate hypothesis directory

In [5]:
def eval_dir(hypdir, querylist, annot_root_1, annot_root_2, hop_sec, savefile = None):
    allErrs = {}
    cnt = 0
    print(f'Processing {hypdir} ', end='')
    with open(querylist, 'r') as f:
        for line in f:
            parts = line.strip().split()
            assert len(parts) == 2
            basename = os.path.basename(parts[0]) + '__' + os.path.basename(parts[1])
            hypfile = hypdir + '/' + basename + '.pkl'
            if not os.path.exists(hypfile):
                print("X", end='')
                continue
            err = eval_file(hypfile, annot_root_1, annot_root_2, hop_sec)
            if err is not None:
                allErrs[basename] = err
            cnt += 1
            if cnt % 500 == 0:
                print(".", end='')
    print(' done')
    if savefile:
        pickle.dump(allErrs, open(savefile, 'wb'))
        
    return allErrs

In [6]:
def eval_file(hypfile, annot_root_1, annot_root_2, hop_sec):
    parts = os.path.basename(hypfile).split('__')
    assert len(parts) == 2
    piece = extractPieceName(parts[0])
    annotfile1 = (annot_root_1 / piece / parts[0]).with_suffix('.beat')
    annotfile2 = (annot_root_2 / piece / parts[1]).with_suffix('.beat')
    
    # if groundtruth annotation files are empty, skip this hypothesis file
    try:
        gt1, gt2 = getTimestamps(annotfile1, annotfile2)
        hypalign = loadAlignment(hypfile) # warping path in frames
    except:
        print(f'Skipping hypothesis file {hypfile}')
        return None

    if hypalign is None:
        err = np.array(np.ones(gt1.shape) * np.inf)
    else:
        pred2 = np.interp(gt1, hypalign[0,:]*hop_sec, hypalign[1,:]*hop_sec)
        err = pred2 - gt2
    return err

In [7]:
def extractPieceName(fullpath):
    basename = os.path.basename(fullpath) # e.g. Chopin_Op068No3_Sztompka-1959_pid9170b-21
    parts = basename.split('_')
    piece = '_'.join(parts[0:2]) # e.g. Chopin_Op068No3
    return piece

In [8]:
def getTimestamps(annotfile1, annotfile2):
    df1 = pd.read_csv(annotfile1, header=None, sep='\s+', skiprows=3) 
    df2 = pd.read_csv(annotfile2, header=None, sep='\s+', skiprows=3)

    df_merged = pd.merge(df1, df2, on=[2], how='inner')
    df_merged = df_merged[df_merged[2].astype(str).str.match(".*\d$")]

    return df_merged['0_x'], df_merged['0_y']

In [9]:
def loadAlignment(hypfile):
    with open(hypfile, 'rb') as f:
        d = pickle.load(f)
    return d

In [ ]:
def eval_all_dirs(rootdir, querylist, hop_sec, outdir, benchmark, system, L=None):
    """Evaluate a system (including sparse_parflex / runtime_for_sparflex with optional L).
    For those systems with chunk length L, hypdir = rootdir/benchmark/<system>/L_<L>.
    For other systems, hypdir = rootdir/benchmark/system."""
    if not os.path.exists(outdir):
        os.mkdir(outdir)
    if benchmark == 'partialOverlap':
        annot_root_1 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Benchmarks/partialStart/annotations_beat')
        annot_root_2 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Benchmarks/partialEnd/annotations_beat')
    elif 'prepost' in benchmark:
        sec = benchmark.split('_')[-1]
        annot_root_1 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Benchmarks/pre_{sec}/annotations_beat')
        annot_root_2 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Benchmarks/post_{sec}/annotations_beat')
    elif benchmark == 'matching':
        annot_root_1 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Modified/annotations_beat')
        annot_root_2 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Modified/annotations_beat')
    else:
        annot_root_1 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Benchmarks/{benchmark}/annotations_beat')
        annot_root_2 = Path(f'/home/ijain/ttmp/Chopin_Mazurkas_Modified/annotations_beat')
    if system in ('sparse_parflex', 'runtime_for_sparflex') and L is not None:
        hypdir = f'{rootdir}/{benchmark}/{system}/L_{L}'
        system_label = f'{system}_L_{L}'
    else:
        hypdir = f'{rootdir}/{benchmark}/{system}'
        system_label = system
    if os.path.isdir(hypdir):
        outpath = outdir + '/' + f'{benchmark}'
        Path(outpath).mkdir(parents=True, exist_ok=True)
        savefile = outpath + '/' + f'{system_label}' + '.pkl'
        allErrs = eval_dir(hypdir, querylist, annot_root_1, annot_root_2, hop_sec, savefile=savefile)

**Evaluate on Data**

In [11]:
EXPERIMENTS_ROOT = f'experiments_{DATASET}/{VERSION}'
hop_sec = 512 * 1 / 22050
outdir = f'evaluations_{DATASET}/{VERSION}'
Path(outdir).mkdir(parents=True, exist_ok=True)

In [12]:
def eval_benchmark(experiments_root, hop_sec, outdir, benchmark, system, L=None):
    eval_all_dirs(experiments_root, query_list, hop_sec, outdir, benchmark, system, L)

In [13]:
# L values for sparse_parflex (saved under experiments_.../benchmark/sparse_parflex/L_<L>/)
from run_sparse_parflex_L_sweep_config import L_VALUES
# Systems to evaluate (as in 04_Align_Benchmarks.ipynb)
# (ParFlexDTW is not included as a separate base system here; each sparse_parflex L is treated as its own system.)
BASE_SYSTEMS = ['dtw1', 'dtw2', 'dtw3', 'subseqdtw1', 'subseqdtw2', 'subseqdtw3', 'nwtw', 'flexdtw']

# File labels for sparse_parflex / runtime_for_sparflex with different L values
SPARSE_PARFLEX_FILE_LABELS = [f'sparse_parflex_L_{L}' for L in L_VALUES]

# Display labels for plotting
BASE_SYSTEMS_DISPLAY = ['DTW1', 'DTW2', 'DTW3', 'SubseqDTW1', 'SubseqDTW2', 'SubseqDTW3', 'NWTW', 'FlexDTW']
SPARSE_PARFLEX_DISPLAY_LABELS = [f'sparse_parflex L={L}' for L in L_VALUES]

SYSTEM_FILE_LABELS = BASE_SYSTEMS + SPARSE_PARFLEX_FILE_LABELS  
SYSTEM_DISPLAY_LABELS = BASE_SYSTEMS_DISPLAY + SPARSE_PARFLEX_DISPLAY_LABELS
BENCHMARKS = ['Matching', 'Subseq_20', 'Subseq_30', 'Subseq_40', 'PartialStart', 'PartialEnd', 'PartialOverlap', 
              'Pre_5', 'Pre_10', 'Pre_20', 'Post_5', 'Post_10', 'Post_20', 'PrePost_5', 
              'PrePost_10', 'PrePost_20']

In [14]:
benchmarks = []
for benchmark in BENCHMARKS:
    if benchmark == 'PartialOverlap':
        benchmarks.append('partialOverlap')
    elif benchmark == 'PartialStart':
        benchmarks.append('partialStart')
    elif benchmark == 'PartialEnd':
        benchmarks.append('partialEnd')
    elif 'PrePost' in benchmark:
        sec = benchmark.split('_')[-1]
        benchmarks.append(f'prepost_{sec}')
    else:
        benchmarks.append(benchmark.lower())

In [15]:
def eval_all_benchmarks(experiments_root, hop_sec, outdir, benchmarks):
    for benchmark in benchmarks:
        # evaluate base systems (no L)
        for system in BASE_SYSTEMS:
            eval_benchmark(experiments_root, hop_sec, outdir, benchmark, system)
        # reference dense parflex (needed for toy parity vs sparse / runtime_for_sparflex)
        if VERSION == 'toy':
            eval_benchmark(experiments_root, hop_sec, outdir, benchmark, 'parflex')
        # evaluate sparse_parflex for each L
        for L in L_VALUES:
            eval_benchmark(experiments_root, hop_sec, outdir, benchmark, 'sparse_parflex', L)


In [ ]:
eval_all_benchmarks(EXPERIMENTS_ROOT, hop_sec, outdir, benchmarks)

Processing experiments_test/full/matching/dtw1 .......Skipping hypothesis file experiments_test/full/matching/dtw1/Chopin_Op068No3_Kiepura-1999_pid9098-07__Chopin_Op068No3_Rangell-2001_pid9094b-27.pkl
 done
Processing experiments_test/full/matching/dtw2 ....... done
Processing experiments_test/full/matching/dtw3 

### Toy: parflex vs sparse_parflex vs runtime_for_sparflex

For `VERSION == 'toy'`, per-query evaluation errors in `parflex.pkl`, `sparse_parflex_L_<L>.pkl`, and `runtime_for_sparflex_L_<L>.pkl` should match (same keys and error arrays) for each benchmark and each `L` in `L_VALUES`.


In [16]:
def _assert_err_dicts_equal(d_ref, d_other, ref_label, other_label):
    if set(d_ref.keys()) != set(d_other.keys()):
        missing = set(d_ref.keys()) ^ set(d_other.keys())
        raise AssertionError(f'{ref_label} vs {other_label}: key mismatch {missing}')
    for k in sorted(d_ref.keys()):
        e1 = np.asarray(d_ref[k])
        e2 = np.asarray(d_other[k])
        if e1.shape != e2.shape:
            raise AssertionError(f'{ref_label} vs {other_label}: shape {k} {e1.shape} vs {e2.shape}')
        if not np.allclose(e1, e2, rtol=0, atol=0, equal_nan=True):
            raise AssertionError(f'{ref_label} vs {other_label}: values differ for {k}')


def check_toy_parflex_sparse_runtime_parity(eval_root, benchmarks, L_values):
    """Raise AssertionError if parflex, sparse_parflex, and runtime_for_sparflex disagree on toy."""
    if VERSION != 'toy':
        print('Skipping parflex/sparse/runtime parity (set VERSION to toy).')
        return
    eval_root = Path(eval_root)
    for benchmark in benchmarks:
        ref_path = eval_root / benchmark / 'parflex.pkl'
        if not ref_path.is_file():
            raise FileNotFoundError(f'Missing reference {ref_path} (run eval with VERSION=toy).')
        with open(ref_path, 'rb') as f:
            ref_errs = pickle.load(f)
        for L in L_values:
            for name in ('sparse_parflex'):
                other_path = eval_root / benchmark / f'{name}_L_{L}.pkl'
                if not other_path.is_file():
                    raise FileNotFoundError(f'Missing {other_path}')
                with open(other_path, 'rb') as f:
                    other = pickle.load(f)
                _assert_err_dicts_equal(ref_errs, other, str(ref_path), str(other_path))
    print('OK: parflex, sparse_parflex, and runtime_for_sparflex evaluation errors match for all benchmarks and L values.')


check_toy_parflex_sparse_runtime_parity(outdir, benchmarks, L_VALUES)


Skipping parflex/sparse/runtime parity (set VERSION to toy).


### Plot error vs tolerance

In [17]:
def calc_error_rates(errFile, maxTol):
    print(errFile)
    # read from file
    with open(errFile, 'rb') as f:
        allErrs = pickle.load(f)
    
    # collect all errors
    errsFlat = []
    # print(allErrs)
    for query in allErrs:
        # print(query)
        errs = np.array(allErrs[query])
        errsFlat.append(errs)
    # print(errsFlat)
    errsFlat = np.concatenate(errsFlat)
    
    # calculate error rates
    errRates = np.zeros(maxTol+1)
    for i in range(maxTol+1):
        errRates[i] = np.mean(np.abs(errsFlat) > i/1000)
    
    return errRates, errsFlat

In [18]:
def calc_error_rates_batch(indir, basenames, maxTol):
    errRates = np.zeros((len(basenames), maxTol+1))
    allErrVals = []
    print('Computing error rates ', end='')
    for i, basename in enumerate(basenames):
        errFile = indir + '/' + basename + '.pkl'
        errRates[i,:], errors = calc_error_rates(errFile, maxTol)
        allErrVals.append(errors)
        print('.', end='')
    print(' done')
    return errRates, allErrVals

In [19]:
def plot_multiple_roc(errRates, basenames):
    numSystems = errRates.shape[0]
    maxTol = errRates.shape[1] - 1
    cmap = cm.get_cmap('tab10')
    dark = 0.72
    for i in range(numSystems):
        r, g, b = to_rgb(cmap(i % 10))
        color = (r * dark, g * dark, b * dark)
        plt.plot(np.arange(maxTol+1), errRates[i,:] * 100.0, color=color)
    plt.legend(basenames, bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
    plt.xlabel('Error Tolerance (ms)')
    plt.ylabel('Error Rate (%)')
    plt.show()
    return

**Evaluate on Data**

In [20]:
EVAL_ROOT_DIR = f'evaluations_{DATASET}/{VERSION}'
toPlot = []

for benchmark in benchmarks:
    for system_label in SYSTEM_FILE_LABELS:
        toPlot.append(f'{benchmark}/{system_label}')
maxTol = 1000 # in msec

In [21]:
def get_errs(eval_root_dir, toPlot, maxTol):
    errRates, errVals = calc_error_rates_batch(EVAL_ROOT_DIR, toPlot, maxTol)
    return errRates, errVals

In [22]:
errRates, errVals = get_errs(EVAL_ROOT_DIR, toPlot, maxTol)

Computing error rates evaluations_test/full/matching/dtw1.pkl
.evaluations_test/full/matching/dtw2.pkl
.evaluations_test/full/matching/dtw3.pkl
.evaluations_test/full/matching/subseqdtw1.pkl
.evaluations_test/full/matching/subseqdtw2.pkl
.evaluations_test/full/matching/subseqdtw3.pkl
.evaluations_test/full/matching/nwtw.pkl
.evaluations_test/full/matching/flexdtw.pkl
.evaluations_test/full/matching/sparse_parflex_L_4000.pkl
.evaluations_test/full/matching/sparse_parflex_L_6000.pkl
.evaluations_test/full/subseq_20/dtw1.pkl
.evaluations_test/full/subseq_20/dtw2.pkl
.evaluations_test/full/subseq_20/dtw3.pkl
.evaluations_test/full/subseq_20/subseqdtw1.pkl
.evaluations_test/full/subseq_20/subseqdtw2.pkl
.evaluations_test/full/subseq_20/subseqdtw3.pkl
.evaluations_test/full/subseq_20/nwtw.pkl
.evaluations_test/full/subseq_20/flexdtw.pkl
.evaluations_test/full/subseq_20/sparse_parflex_L_4000.pkl
.evaluations_test/full/subseq_20/sparse_parflex_L_6000.pkl
.evaluations_test/full/subseq_30/dtw1.p

In [23]:
errRates.shape[0] == len(SYSTEM_FILE_LABELS) * len(BENCHMARKS)

True

In [24]:
np.save(f'evaluations_{DATASET}/{VERSION}_errRates', errRates)

### Make Plots (New)

In [47]:
def make_df(time, errRates):
    data = {}
    
    num_systems = len(SYSTEM_DISPLAY_LABELS)
    num_benchmarks = len(BENCHMARKS)
    
    data['Benchmark'] = []
    for benchmark in BENCHMARKS:
        data['Benchmark'] += [benchmark] * num_systems
    
    data['System'] = [label for label in SYSTEM_DISPLAY_LABELS] * num_benchmarks

    data['Error'] = []
    for i in range(num_benchmarks * num_systems):
        data['Error'].append(errRates[i][time]*100)
            
    df = pd.DataFrame.from_dict(data)
    benchmark_categories = CategoricalDtype(categories=BENCHMARKS, ordered=True)
    df.Benchmark = df.Benchmark.astype(benchmark_categories)
    system_categories = CategoricalDtype(categories=SYSTEM_DISPLAY_LABELS, ordered=True)
    df['System'] = df['System'].astype(system_categories)
    
    return df

In [48]:
errRates = np.load(f'evaluations_{DATASET}/{VERSION}_errRates.npy')

In [49]:
# Error rates at 200ms tolerance
ms200_df = make_df(200, errRates)

PLOT_SYSTEMS = [
    'DTW1', 'DTW2', 'DTW3',
    'SubseqDTW1', 'SubseqDTW2', 'SubseqDTW3',
    'NWTW', 'FlexDTW', 'ParFlexDTW'
]


def prep_plot_df(df):
    plot_df = df[df['System'].isin(PLOT_SYSTEMS + ['sparse_parflex L=4000'])].copy()

    # `System` is a pandas Categorical in `make_df`; cast to string before renaming
    # so introducing the new label 'ParFlexDTW' doesn't become NaN.
    plot_df['System'] = plot_df['System'].astype(str)
    plot_df.loc[plot_df['System'] == 'sparse_parflex L=4000', 'System'] = 'ParFlexDTW'

    plot_df['System'] = pd.Categorical(plot_df['System'], categories=PLOT_SYSTEMS, ordered=True)
    return plot_df

In [50]:
# Error rates at 100ms and 500ms tolerance (black horizontal bars on plot)
ms100_df = make_df(100, errRates)
ms500_df = make_df(500, errRates)

ms200_plot_df = prep_plot_df(ms200_df)
ms100_plot_df = prep_plot_df(ms100_df)
ms500_plot_df = prep_plot_df(ms500_df)

In [51]:
plot_colors = {
    # DTW → greens (shifted darker for visibility)
    'DTW1': '#8FE88B',   # lighter + softer → clearer jump from DTW2
  
    'DTW2': '#2EA12E',   # mid green, clearly below DTW3
    'DTW3': '#145A14',   # darkest green

    # SubseqDTW → blues (same spacing idea)
    'SubseqDTW1': '#9AC4FF',  # lighter → more distinct from mid blue
    'SubseqDTW2': '#2F80ED',
    'SubseqDTW3': '#1C3F99',

    # Others
    'NWTW': '#C77DFF',
    'FlexDTW': '#6B7280',
    'ParFlexDTW': '#FF3B30',
}

In [52]:
(ggplot(ms200_plot_df, aes(x="System", y="Error", fill="System")) +
    geom_bar(width=0.7,
             position=position_dodge2(preserve='single', width=0.25),
             stat='identity') +
    scale_y_continuous(expand=(0, 0), limits=(0, 100)) +
    scale_fill_manual(values=plot_colors, breaks=PLOT_SYSTEMS) +
    geom_crossbar(aes(ymin="Error", ymax="Error"),
                  data=ms100_plot_df, width=0.5, color="black") +
    geom_crossbar(aes(ymin="Error", ymax="Error"),
                  data=ms500_plot_df, width=0.5, color="black") +
    facet_grid('. ~ Benchmark') +
    theme_bw() +
    labs(y="Error Rate (%)") +
    theme(
        dpi=600,
        figure_size=(15, 5),
        legend_position='bottom',
        legend_direction="horizontal",
        legend_box="horizontal",
        legend_background=element_blank(),
        legend_title=element_text(size=10),
        legend_text=element_text(size=8),
        strip_background=element_blank(),
        strip_text_x=element_text(size=9, weight='bold'),
        axis_text_x=element_blank(),
        axis_ticks_x=element_blank(),
        axis_text_y=element_text(size=6, colour='black'),
        axis_title_x=element_blank(),
        axis_title_y=element_text(size=10, margin={'r': 6.0})
    ) +
    guides(fill=guide_legend(
        nrow=1, byrow=True, title="",
        override_aes={'size': 0}
    ))
)
